In [ ]:
import os
import sys

import numpy as np
import polars as pl
from tqdm import trange

sys.path.append("/workspace")

from drsd.reader import MINDsmallReader

pl.Config(tbl_rows=5)
os.makedirs("/workspace/processed", exist_ok=True)

In [ ]:
def batch_apply(df: pl.DataFrame, func, batch_size: int) -> pl.DataFrame:
    return pl.concat([func(df[i : i + batch_size]) for i in trange(0, len(df), batch_size)])

In [ ]:
reader = MINDsmallReader()

news_df = pl.concat([reader.get_news_df("train"), reader.get_news_df("dev")]).unique().sort("news_id")
assert news_df.get_column("news_id").is_unique().all()

behavior_df = pl.concat([reader.get_behavior_df("train"), reader.get_behavior_df("dev")]).with_columns(
    pl.col("time").str.strptime(pl.Datetime, "%m/%d/%Y %r")
)

display(news_df, behavior_df)

In [ ]:
news_embedding_df = pl.read_parquet("/workspace/processed/news_embedding.parquet")
news_entity_df = pl.read_parquet("/workspace/processed/news_entity.parquet")
news_time_df = pl.read_parquet("/workspace/processed/news_time.parquet")
display(news_embedding_df, news_entity_df, news_time_df)

### Initialize

In [ ]:
feature_df = (
    behavior_df.select(
        pl.col("history").list.last().alias("news_id"),
        pl.concat_list(pl.col("clicked"), pl.col("non_clicked")).alias("candidate_news_id"),
    )
    .filter(pl.col("news_id").is_not_null())
    .explode("candidate_news_id")
    .unique()
    .sort("news_id", "candidate_news_id")
)
feature_df

### embedding

In [ ]:
feature_df = feature_df.join(news_embedding_df, left_on="news_id", right_on="news_id", how="left", suffix="").join(
    news_embedding_df, left_on="candidate_news_id", right_on="news_id", how="left", suffix="_candidate"
)
feature_df = batch_apply(
    feature_df,
    lambda df: df.with_columns(
        pl.Series(df["embedding_pca"].to_numpy() - df["embedding_pca_candidate"].to_numpy()).alias(
            "embedding_pca_diff"
        ),
        pl.Series(
            (df["embedding"].to_numpy() * df["embedding_candidate"].to_numpy()).sum(axis=1)
            / np.linalg.norm(df["embedding"].to_numpy(), axis=1)
            / np.linalg.norm(df["embedding_candidate"].to_numpy(), axis=1)
        ).alias("embedding_sim"),
    ),
    100000,
).drop("embedding", "embedding_candidate")
feature_df

### entity

In [ ]:
feature_df = batch_apply(
    feature_df,
    lambda df: (
        df.join(news_entity_df, left_on="news_id", right_on="news_id", how="left", suffix="")
        .join(news_entity_df, left_on="candidate_news_id", right_on="news_id", how="left", suffix="_candidate")
        .with_columns(
            (
                pl.col("entities").list.set_intersection(pl.col("entities_candidate")).list.len()
                / pl.max_horizontal(pl.col("entities").list.len(), pl.col("entities_candidate").list.len())
            ).alias("entity_overlap")
        )
        .drop("entities", "entities_candidate")
    ),
    100000,
)
feature_df

### category

In [ ]:
feature_df = batch_apply(
    feature_df,
    lambda df: (
        df.join(
            news_df.select("news_id", "category", "subcategory"),
            left_on="news_id",
            right_on="news_id",
            how="left",
            suffix="",
        )
        .join(
            news_df.select("news_id", "category", "subcategory"),
            left_on="candidate_news_id",
            right_on="news_id",
            how="left",
            suffix="_candidate",
        )
        .with_columns(
            (pl.col("category") + "_" + pl.col("category_candidate")).alias("category_concat"),
            (
                pl.col("category")
                + "-"
                + pl.col("subcategory")
                + "_"
                + pl.col("subcategory")
                + "-"
                + pl.col("subcategory_candidate")
            ).alias("subcategory_concat"),
            (pl.col("category") == pl.col("category_candidate")).alias("category_match"),
            (
                (pl.col("category") == pl.col("category_candidate"))
                & (pl.col("subcategory") == pl.col("subcategory_candidate"))
            ).alias("subcategory_match"),
        )
    ),
    100000,
)
feature_df

In [ ]:
feature_df = (
    feature_df.join(news_time_df, left_on="news_id", right_on="news_id", how="left", suffix="")
    .join(news_time_df, left_on="candidate_news_id", right_on="news_id", how="left", suffix="_candidate")
    .with_columns((pl.col("time_candidate") - pl.col("time")).dt.total_seconds().alias("time_diff"))
    .drop("time", "time_candidate")
)
feature_df

### Output DataFrame

In [ ]:
# unnest embedding
feature_df = feature_df.with_columns(
    pl.col("embedding_pca").arr.to_struct(fields=lambda idx: f"embedding_pca_{idx:02}"),
    pl.col("embedding_pca_candidate").arr.to_struct(fields=lambda idx: f"embedding_pca_candidate_{idx:02}"),
    pl.col("embedding_pca_diff").arr.to_struct(fields=lambda idx: f"embedding_diff_{idx:02}"),
).unnest("embedding_pca", "embedding_pca_candidate", "embedding_pca_diff")
feature_df

In [ ]:
feature_df.write_parquet("/workspace/processed/feature.parquet")
feature_df